# 02 — Train v3: Mô hình hoá

**GĐ 3** theo đề bài: Train ≥ 2 model, split Train/Val/Test, có đồ thị, accuracy ≥ 85%.

Pipeline:
1. Setup + path
2. Resize toàn bộ raw → processed (224×224) + split 70/15/15
3. Visualize sample + augmentation demo
4. Train MobileNetV2 (2-phase: freeze → fine-tune)
5. Train ResNet50 (2-phase)
6. Đánh giá test set + so sánh
7. Lưu metrics + biểu đồ

> Mọi thao tác chạy trong notebook, kết quả hiển thị tại chỗ + lưu `results/*.png`, `checkpoints/*.keras`.

## 1. Setup + path

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

# CWD về project root để import được module
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == 'notebook':
    PROJECT_DIR = PROJECT_DIR.parent
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

RAW_DIR        = PROJECT_DIR / 'dataset' / 'raw'
PROCESSED_DIR  = PROJECT_DIR / 'dataset' / 'processed_imgs'
TRAIN_DIR      = PROJECT_DIR / 'dataset' / 'train'
VALID_DIR      = PROJECT_DIR / 'dataset' / 'valid'
TEST_DIR       = PROJECT_DIR / 'dataset' / 'test'
CKPT_DIR       = PROJECT_DIR / 'checkpoints'
RESULTS_DIR    = PROJECT_DIR / 'results'
for d in (CKPT_DIR, RESULTS_DIR):
    d.mkdir(exist_ok=True)

IMG_SIZE   = 224
BATCH_SIZE = 32
SEED       = 42

import tensorflow as tf
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)
print('GPU       :', tf.config.list_physical_devices('GPU') or 'CPU only')

## 2. Resize + split 70/15/15

Reuse `preprocessing/preprocess.py` (đã có hàm `preprocess_all` + `split_dataset`).
Skip nếu đã split rồi (có thể force lại bằng `FORCE_SPLIT = True`).

In [ ]:
from preprocessing.preprocess import preprocess_all, split_dataset

FORCE_SPLIT = False  # đặt True nếu muốn re-split

need_split = FORCE_SPLIT or not TRAIN_DIR.exists() or not any(TRAIN_DIR.iterdir())
if need_split:
    print('[1/2] Resize 224x224 ...')
    summary = preprocess_all(RAW_DIR, PROCESSED_DIR, img_size=IMG_SIZE)
    print('[2/2] Split 70/15/15 ...')
    split_dataset(PROCESSED_DIR, PROJECT_DIR / 'dataset', ratios=(0.7, 0.15, 0.15))
else:
    print('[skip] đã có sẵn dataset/train|valid|test  (set FORCE_SPLIT=True để re-split)')

# Thống kê số lượng từng split
rows = []
for split, d in [('train', TRAIN_DIR), ('valid', VALID_DIR), ('test', TEST_DIR)]:
    if not d.exists():
        continue
    for cls in sorted(p for p in d.iterdir() if p.is_dir()):
        rows.append({'split': split, 'class': cls.name,
                     'n': sum(1 for _ in cls.iterdir())})
df_split = pd.DataFrame(rows)
summary_split = df_split.pivot_table(index='class', columns='split', values='n', aggfunc='sum').fillna(0).astype(int)
summary_split.loc['TOTAL'] = summary_split.sum()
summary_split

In [ ]:
# Vẽ phân bố train/valid/test
fig, ax = plt.subplots(figsize=(11, 5))
data = summary_split.drop('TOTAL').sort_index()
data.plot(kind='bar', stacked=True, ax=ax,
          color=['#1976d2', '#fb8c00', '#43a047'], edgecolor='black')
ax.set_title('Phân bố Train / Valid / Test theo class')
ax.set_ylabel('Số ảnh'); ax.set_xticklabels(ax.get_xticklabels(), rotation=70, fontsize=8)
ax.legend(title='split')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'split_distribution.png', bbox_inches='tight')
plt.show()

## 3. Generators + augmentation demo

In [ ]:
from preprocessing.augmentation import build_train_generator, build_eval_generator

train_gen = build_train_generator(TRAIN_DIR, img_size=IMG_SIZE, batch_size=BATCH_SIZE)
valid_gen = build_eval_generator(VALID_DIR, img_size=IMG_SIZE, batch_size=BATCH_SIZE)
test_gen  = build_eval_generator(TEST_DIR,  img_size=IMG_SIZE, batch_size=BATCH_SIZE)

CLASS_NAMES = list(train_gen.class_indices.keys())
NUM_CLASSES = len(CLASS_NAMES)
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')
print(f'Train: {train_gen.samples}  Valid: {valid_gen.samples}  Test: {test_gen.samples}')

In [ ]:
# Augmentation demo — 1 ảnh gốc + 7 phiên bản augment
x_batch, y_batch = next(train_gen)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(x_batch[i])
    cls = CLASS_NAMES[int(np.argmax(y_batch[i]))]
    ax.set_title(cls, fontsize=9); ax.axis('off')
plt.suptitle('Augmentation samples (rotation/flip/zoom/brightness/shift)', fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'augmentation_demo.png', bbox_inches='tight')
plt.show()
train_gen.reset()  # reset để bắt đầu epoch mới

## 4. Train MobileNetV2 (2-phase)

- Phase 1: freeze base, train classifier head (lr=1e-3).
- Phase 2: unfreeze 30 layer cuối, fine-tune (lr=1e-5).

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from models.mobilenet_model import build_mobilenet, unfreeze_for_finetune as unfreeze_mbn

EPOCHS_PHASE1 = 12   # giảm xuống nếu CPU, tăng lên cho GPU
EPOCHS_PHASE2 = 8

def make_callbacks(name):
    return [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7),
        ModelCheckpoint(str(CKPT_DIR / f'{name}_best.keras'),
                        monitor='val_accuracy', save_best_only=True),
    ]

mbn = build_mobilenet(num_classes=NUM_CLASSES, img_size=IMG_SIZE)
mbn.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy',
            metrics=['accuracy'])
mbn.summary(line_length=100)

In [ ]:
# Phase 1 — freeze base, train head
t0 = time.time()
hist1_mbn = mbn.fit(train_gen, validation_data=valid_gen,
                    epochs=EPOCHS_PHASE1, callbacks=make_callbacks('mobilenet'), verbose=2)
print(f'[Phase 1] {time.time()-t0:.0f}s')

In [ ]:
# Phase 2 — unfreeze 30 layer, fine-tune
unfreeze_mbn(mbn, n_layers=30)
mbn.compile(optimizer=Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
t0 = time.time()
hist2_mbn = mbn.fit(train_gen, validation_data=valid_gen,
                    epochs=EPOCHS_PHASE2, callbacks=make_callbacks('mobilenet'), verbose=2)
print(f'[Phase 2] {time.time()-t0:.0f}s')

def merge_hist(h1, h2):
    out = {}
    for k in h1.history:
        out[k] = list(h1.history[k]) + list(h2.history.get(k, []))
    return out
hist_mbn = merge_hist(hist1_mbn, hist2_mbn)

## 5. Train ResNet50 (2-phase)

In [ ]:
from models.resnet_model import build_resnet, unfreeze_for_finetune as unfreeze_rn

rn = build_resnet(num_classes=NUM_CLASSES, img_size=IMG_SIZE)
rn.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])

t0 = time.time()
hist1_rn = rn.fit(train_gen, validation_data=valid_gen,
                  epochs=EPOCHS_PHASE1, callbacks=make_callbacks('resnet'), verbose=2)
print(f'[Phase 1] {time.time()-t0:.0f}s')

unfreeze_rn(rn, n_layers=40)
rn.compile(optimizer=Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
t0 = time.time()
hist2_rn = rn.fit(train_gen, validation_data=valid_gen,
                  epochs=EPOCHS_PHASE2, callbacks=make_callbacks('resnet'), verbose=2)
print(f'[Phase 2] {time.time()-t0:.0f}s')
hist_rn = merge_hist(hist1_rn, hist2_rn)

## 6. Đánh giá test set + so sánh

- Training curves (loss/accuracy, train vs val) cho cả 2 model.
- Đánh giá final trên test set: accuracy, precision, recall, F1.
- Bar chart so sánh.

In [ ]:
# Training curves
def plot_curves(hist, title, ax_loss, ax_acc):
    epochs = np.arange(1, len(hist['loss']) + 1)
    ax_loss.plot(epochs, hist['loss'],     label='train', color='#1976d2')
    ax_loss.plot(epochs, hist['val_loss'], label='val',   color='#c62828')
    ax_loss.axvline(EPOCHS_PHASE1 + 0.5, color='gray', ls=':', label='fine-tune')
    ax_loss.set_title(f'{title} — Loss'); ax_loss.set_xlabel('epoch'); ax_loss.legend()
    ax_acc.plot(epochs, hist['accuracy'],     label='train', color='#1976d2')
    ax_acc.plot(epochs, hist['val_accuracy'], label='val',   color='#c62828')
    ax_acc.axvline(EPOCHS_PHASE1 + 0.5, color='gray', ls=':')
    ax_acc.set_title(f'{title} — Accuracy'); ax_acc.set_xlabel('epoch'); ax_acc.legend()

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
plot_curves(hist_mbn, 'MobileNetV2', axes[0,0], axes[0,1])
plot_curves(hist_rn,  'ResNet50',    axes[1,0], axes[1,1])
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'training_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# Đánh giá test set
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              classification_report, confusion_matrix)

def eval_on_test(model, name):
    test_gen.reset()
    y_true = test_gen.classes
    y_prob = model.predict(test_gen, verbose=0)
    y_pred = y_prob.argmax(axis=1)
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0)
    print(f'\n=== {name} ===')
    print(f'Accuracy : {acc*100:.2f}%')
    print(f'Precision: {prec*100:.2f}%  Recall: {rec*100:.2f}%  F1: {f1*100:.2f}%')
    return {'name': name, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
            'y_true': y_true, 'y_pred': y_pred, 'y_prob': y_prob}

res_mbn = eval_on_test(mbn, 'MobileNetV2')
res_rn  = eval_on_test(rn,  'ResNet50')

In [ ]:
# Bar chart so sánh + confusion matrices
metrics = ['accuracy', 'precision', 'recall', 'f1']
x = np.arange(len(metrics)); w = 0.35
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
ax.bar(x - w/2, [res_mbn[m] for m in metrics], w, label='MobileNetV2', color='#1976d2')
ax.bar(x + w/2, [res_rn[m]  for m in metrics], w, label='ResNet50',    color='#c62828')
ax.axhline(0.85, color='green', ls='--', label='target 85%')
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05); ax.set_title('So sánh metrics (test set)'); ax.legend()
for i, m in enumerate(metrics):
    ax.text(i - w/2, res_mbn[m] + 0.01, f"{res_mbn[m]*100:.1f}%", ha='center', fontsize=8)
    ax.text(i + w/2, res_rn[m]  + 0.01, f"{res_rn[m]*100:.1f}%",  ha='center', fontsize=8)

for ax, res in zip(axes[1:], [res_mbn, res_rn]):
    cm = confusion_matrix(res['y_true'], res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cbar=False,
                annot_kws={'fontsize': 7})
    ax.set_title(f"Confusion Matrix — {res['name']}")
    ax.set_xlabel('predicted'); ax.set_ylabel('true')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=70, fontsize=7)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0,  fontsize=7)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'comparison_and_cm.png', bbox_inches='tight')
plt.show()

# Classification reports
for res in (res_mbn, res_rn):
    print(f"\n--- {res['name']} ---")
    print(classification_report(res['y_true'], res['y_pred'],
                                target_names=CLASS_NAMES, zero_division=0))

## 7. Lưu metrics ra JSON cho báo cáo

In [ ]:
for res, ckpt_name in [(res_mbn, 'mobilenet'), (res_rn, 'resnet')]:
    out = RESULTS_DIR / f'{ckpt_name}_metrics.json'
    with open(out, 'w', encoding='utf-8') as f:
        json.dump({
            'name': res['name'],
            'accuracy': float(res['accuracy']),
            'precision_macro': float(res['precision']),
            'recall_macro':    float(res['recall']),
            'f1_macro':        float(res['f1']),
            'class_names':     CLASS_NAMES,
            'checkpoint':      str(CKPT_DIR / f'{ckpt_name}_best.keras'),
        }, f, indent=2, ensure_ascii=False)
    print(f'saved -> {out}')

best_name = res_mbn['name'] if res_mbn['accuracy'] >= res_rn['accuracy'] else res_rn['name']
best_acc  = max(res_mbn['accuracy'], res_rn['accuracy'])
print(f'\nModel tốt nhất: {best_name}  acc = {best_acc*100:.2f}%')
print(f'Đạt yêu cầu ≥ 85%: {best_acc >= 0.85}')

## 8. Grad-CAM — Vùng model 'chú ý' khi dự đoán

**Grad-CAM (Gradient-weighted Class Activation Mapping)** là kỹ thuật visualize **vùng nào trên ảnh** mà CNN tập trung vào khi đưa ra dự đoán. Nguyên lý:

1. Forward pass → predict class_idx
2. Lấy gradient của `class_score` so với feature map của lớp conv cuối
3. Trung bình gradient theo spatial → trọng số quan trọng cho mỗi channel
4. Weighted sum của feature maps → heatmap 7×7, resize lên 224×224 đè lên ảnh gốc

**Mục đích:**
- Verify model học **đặc trưng có ý nghĩa** (vỏ thâm, đốm hỏng) hay 'cheat' theo background
- Giải thích quyết định để báo cáo có insight
- Đáp ứng yêu cầu GĐ 4 trong CLAUDE.md

In [ ]:
# === Grad-CAM helper functions ===
import tensorflow as tf
import cv2

def find_last_conv_layer(base_model):
    """Tự tìm conv layer cuối cùng có 4D output (batch, h, w, c)."""
    for layer in reversed(base_model.layers):
        try:
            shape = layer.output.shape
            if len(shape) == 4 and shape[1] is not None and shape[1] > 1:
                return layer.name
        except AttributeError:
            continue
    raise ValueError('Không tìm được conv layer 4D')

def make_gradcam_heatmap(model, img_array, class_idx, last_conv_name):
    """Tính Grad-CAM heatmap cho 1 ảnh."""
    base = next((l for l in model.layers if isinstance(l, tf.keras.Model)), None)
    if base is None:
        raise ValueError('Không tìm thấy backbone')
    last_conv = base.get_layer(last_conv_name)
    grad_model = tf.keras.models.Model(
        inputs=model.input,
        outputs=[last_conv.output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, preds = grad_model(img_array, training=False)
        class_score = preds[:, class_idx]
    grads = tape.gradient(class_score, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap).numpy()
    heatmap = np.maximum(heatmap, 0) / (heatmap.max() + 1e-8)
    return heatmap

def visualize_gradcam(model, test_gen, class_names, model_name,
                       n_samples=8, save_path=None):
    """Visualize Grad-CAM cho n_samples ảnh từ test_gen."""
    # Auto detect last conv
    base = next((l for l in model.layers if isinstance(l, tf.keras.Model)), None)
    last_conv_name = find_last_conv_layer(base)
    print(f'[{model_name}] Last conv layer: {last_conv_name}')
    
    test_gen.reset()
    x_batch, y_batch = next(test_gen)
    
    fig, axes = plt.subplots(2, n_samples, figsize=(n_samples * 2.5, 6))
    for i in range(n_samples):
        img = x_batch[i]
        true_idx = int(np.argmax(y_batch[i]))
        pred = model.predict(img[np.newaxis], verbose=0)[0]
        pred_idx = int(np.argmax(pred))
        is_correct = (pred_idx == true_idx)
        
        heatmap = make_gradcam_heatmap(model, img[np.newaxis], pred_idx, last_conv_name)
        heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
        
        # Normalize ảnh để display (preprocessed có thể negative)
        img_disp = (img - img.min()) / (img.max() - img.min() + 1e-8)
        
        axes[0, i].imshow(img_disp)
        axes[0, i].set_title(f'True: {class_names[true_idx]}', fontsize=8,
                              color='green' if is_correct else 'red')
        axes[0, i].axis('off')
        
        axes[1, i].imshow(img_disp)
        axes[1, i].imshow(heatmap, cmap='jet', alpha=0.5)
        axes[1, i].set_title(f'Pred: {class_names[pred_idx]}\n'
                              f'({pred[pred_idx]*100:.1f}%) {"✓" if is_correct else "✗"}',
                              fontsize=8)
        axes[1, i].axis('off')
    
    plt.suptitle(f'Grad-CAM — {model_name}: vùng model chú ý khi dự đoán',
                 fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=130)
        print(f'Saved -> {save_path}')
    plt.show()

In [ ]:
# Grad-CAM cho cả 2 model
print('Generating Grad-CAM for MobileNetV2...')
visualize_gradcam(
    mbn, mbn_test_gen if 'mbn_test_gen' in dir() else test_gen,
    CLASS_NAMES, 'MobileNetV2',
    n_samples=8, save_path=RESULTS_DIR / 'gradcam_mobilenet.png'
)

print('\nGenerating Grad-CAM for ResNet50...')
visualize_gradcam(
    rn, rn_test_gen, CLASS_NAMES, 'ResNet50',
    n_samples=8, save_path=RESULTS_DIR / 'gradcam_resnet.png'
)

## 9. Misclassified Samples — Phân tích trường hợp dự đoán SAI

Hiển thị các ảnh model dự đoán SAI để:
- Hiểu tại sao 7-10% test set bị sai
- Phát hiện pattern lỗi (cùng loại quả Fresh↔Rotten? Khác loại quả cùng màu?)
- Đề xuất hướng cải thiện cho phần 'Hạn chế và hướng phát triển' trong báo cáo

In [ ]:
# Misclassified samples cho ResNet50 (model tốt nhất)
y_true = res_rn['y_true']
y_pred = res_rn['y_pred']
y_prob = res_rn['y_prob']
wrong_idx = np.where(y_true != y_pred)[0]
print(f'ResNet50 đoán sai: {len(wrong_idx)}/{len(y_true)} ảnh ({len(wrong_idx)/len(y_true)*100:.2f}%)')

# Sort theo confidence của prediction sai (cao nhất = lỗi 'tự tin nhất' — đáng phân tích nhất)
wrong_conf = y_prob[wrong_idx, y_pred[wrong_idx]]
sorted_idx = wrong_idx[np.argsort(-wrong_conf)]
n_show = min(8, len(sorted_idx))
selected = sorted_idx[:n_show]

# Reload ảnh từ test_gen (không random nên index ổn định)
rn_test_gen.reset()
all_imgs = []
for x, _ in rn_test_gen:
    all_imgs.append(x)
    if sum(len(b) for b in all_imgs) >= len(y_true):
        break
all_imgs = np.concatenate(all_imgs)[:len(y_true)]

fig, axes = plt.subplots(2, n_show, figsize=(n_show * 2.3, 5.5))
for col, idx in enumerate(selected):
    img = all_imgs[idx]
    img_disp = (img - img.min()) / (img.max() - img.min() + 1e-8)
    
    axes[0, col].imshow(img_disp)
    axes[0, col].set_title(f'TRUE:\n{CLASS_NAMES[y_true[idx]]}', fontsize=8, color='green')
    axes[0, col].axis('off')
    
    # Bar chart top-3 confidence
    top3 = np.argsort(y_prob[idx])[-3:][::-1]
    colors = ['#c62828' if i == y_pred[idx] else '#1976d2' for i in top3]
    axes[1, col].barh([CLASS_NAMES[i][:12] for i in top3][::-1],
                       [y_prob[idx, i]*100 for i in top3][::-1],
                       color=colors[::-1])
    axes[1, col].set_xlim(0, 100)
    axes[1, col].set_title(f'PRED (sai):\n{CLASS_NAMES[y_pred[idx]][:14]}\n{wrong_conf[np.argsort(-wrong_conf)[col]]*100:.1f}%',
                            fontsize=7, color='red')
    axes[1, col].tick_params(labelsize=6)

plt.suptitle('Misclassified Samples — ResNet50 (8 lỗi tự tin nhất)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'misclassified_resnet.png', bbox_inches='tight', dpi=130)
plt.show()

# Phân tích pattern lỗi
print('\n=== Pattern lỗi top-10 ===')
wrong_pairs = pd.DataFrame({
    'true':  [CLASS_NAMES[i] for i in y_true[wrong_idx]],
    'pred':  [CLASS_NAMES[i] for i in y_pred[wrong_idx]],
})
common_mistakes = (wrong_pairs.groupby(['true', 'pred']).size()
                              .reset_index(name='count')
                              .sort_values('count', ascending=False))
print(common_mistakes.head(10).to_string(index=False))